### Search Engine With Tools And Agents

In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
## Arxiv--Research
## Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper


In [2]:
## Ued the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name


'wikipedia'

In [3]:
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=250)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
print(arxiv.name)


arxiv


In [4]:
tools=[wiki,arxiv]

In [ ]:
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools.retriever import create_retriever_tool

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

In [8]:
import os
from dotenv import load_dotenv

# Document loading, splitting, HuggingFace embeddings & FAISS
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from langchain_community.vectorstores import FAISS

# Tools (Retriever, Wikipedia, Arxiv)
from langchain_core.tools.retriever import create_retriever_tool
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
from langchain_community.tools import WikipediaQueryRun, ArxivQueryRun

# 1. Load your free HuggingFace token from .env
load_dotenv()

# 2. Load & split LangSmith docs
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)

# 3. Create FAISS vector database using local HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectordb = FAISS.from_documents(documents, embeddings)
retriever = vectordb.as_retriever()

# 4. Create the Retriever, Wikipedia, and Arxiv tools
retriever_tool = create_retriever_tool(
    retriever,
    "langsmith-search",
    "Search any information about Langsmith"
)

wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200))
arxiv = ArxivQueryRun(api_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200))

tools = [wiki, arxiv, retriever_tool]

# 5. Initialize a free HuggingFace Chat LLM (no AgentRouter or OpenAI needed)
endpoint = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
)
llm = ChatHuggingFace(llm=endpoint)

print("100% HuggingFace setup ready! Tools:", [t.name for t in tools])


c:\Users\Haemimont\Videos\gen ai\Q&A ChatBot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3019.85it/s]


100% HuggingFace setup ready! Tools: ['wikipedia', 'arxiv', 'langsmith-search']


In [17]:
## Run all this tools with Agents and LLM Models

## Tools, LLM-->AgentExecutor
from dotenv import load_dotenv
from langchain_groq import ChatGroq
import os

load_dotenv()


groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b")



In [18]:
## Prompt Template
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])
prompt.messages


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [ ]:
## Agents
from langchain_classic.agents import create_openai_tools_agent, AgentExecutor
agent = create_openai_tools_agent(llm, tools, prompt)
## Agent Executor (Next step in tutorial)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor.invoke({"input": "Tell me about Langsmith"})




> Entering new AgentExecutor chain...

Invoking: `langsmith-search` with `{'query': 'Langsmith'}`


LangSmith Observability - Docs by LangChain

root cause, and resolve them with LangSmith Engine.If your applications call models through the LLM Gateway, the gateway traces every call itself, with no tracing setup in your code. For where those traces appear and what they record, see View LLM Gateway traces.For terminology and core concepts, refer to Observability concepts. For trace pricing, retention, and limits, see Usage and billing.To set up a LangSmith instance, visit the Platform setup section to choose between cloud, hybrid, or self-hosted. All options include observability, evaluation, prompt engineering, and deployment.

Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chanc

{'input': 'Tell me about Langsmith',
 'output': '**LangSmith** is the observability, evaluation, and deployment platform built by the creators of LangChain. It gives developers a unified way to instrument, monitor, and improve large‑language‑model (LLM) applications across the entire workflow—from prompting and token usage to downstream agent behavior.\n\n| Feature | What it does | How it’s used |\n|---------|--------------|---------------|\n| **Observability** | Records every LLM call, chain, or agent step as a trace. | Attach a LangSmith client to your code, or let the LLM Gateway auto‑trace when you call models through the gateway. |\n| **Debugging & Monitoring** | Visual dashboards show latency, token counts, and error rates. You can drill into individual traces to see the prompt, the model response, and any downstream logic. | Use the web UI (smith.langchain.com) to filter traces by time, model, or custom tags. |\n| **Evaluation** | Build evaluation datasets from real‑world traces

In [20]:
agent_executor.invoke({"input": "what is ml?"})



> Entering new AgentExecutor chain...
**ML** can refer to a couple of common things, depending on the context:

| Common Meaning | What it is | Typical context |
|----------------|------------|-----------------|
| **Machine Learning** | A field of artificial intelligence that focuses on building algorithms that can learn patterns from data and improve over time without being explicitly programmed for each task. | Data science, AI, predictive modeling, computer vision, natural language processing, etc. |
| **Milliliter** | A unit of volume equal to one thousandth of a liter (≈0.0338 fluid ounces). | Chemistry, medicine, cooking, scientific measurements. |

If you had a different “ML” in mind, let me know!

> Finished chain.


{'input': 'what is ml?',
 'output': '**ML** can refer to a couple of common things, depending on the context:\n\n| Common Meaning | What it is | Typical context |\n|----------------|------------|-----------------|\n| **Machine Learning** | A field of artificial intelligence that focuses on building algorithms that can learn patterns from data and improve over time without being explicitly programmed for each task. | Data science, AI, predictive modeling, computer vision, natural language processing, etc. |\n| **Milliliter** | A unit of volume equal to one thousandth of a liter (≈0.0338\u202ffluid ounces). | Chemistry, medicine, cooking, scientific measurements. |\n\nIf you had a different “ML” in mind, let me know!'}

In [21]:
loader = WebBaseLoader("https://medium.com/@musabzafar63/from-vector-rag-to-materialized-state-rethinking-how-agents-remember-82114ee5a0e3")
docs = loader.load()

# 2. Split and store in FAISS with HuggingFace embeddings
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents, embeddings)
retriever = vectordb.as_retriever()

# 3. Update retriever tool description to match your article
medium_tool = create_retriever_tool(
    retriever,
    "medium-article-search",
    "Search information from Musab Zafar's Medium article about Vector RAG and Materialized State."
)

# 4. Rebuild agent with the new tool and ask your question!
tools = [wiki, arxiv, medium_tool]
agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({"input": "What is the main argument of the Medium article on materialized state?"})




> Entering new AgentExecutor chain...

Invoking: `medium-article-search` with `{'query': 'Medium article on materialized state'}`


= Fold(Previous State, New Event)Memory architectures must operate identically:The Interaction Stream (Write-Ahead Log): The raw back-and-forth turns are recorded as raw chronological events.The Compaction Engine (Background Compiler): An out-of-band background process continuously digests those events. It resolves anaphora (it — >database_port), identifies mutations (overwrite 5432 with 5433), and prunes ephemeral noise.The Materialized State (Core Memory): The distilled state is saved as a structured, declarative document — a clean JSON schema, property graph, or concise markdown profile.When the agent starts a turn, it doesn’t run vector similarity over 50 past conversations. It loads the compiled state directly into its context.Press enter or click to view image in full sizeThe Hidden Engineering Bottleneck Cache EconomicsOnce you accept that Memory 

{'input': 'What is the main argument of the Medium article on materialized state?',
 'output': '**Main argument**\n\nThe article argues that the way most modern LLM‑powered agents “remember” is fundamentally flawed.  \nInstead of treating past conversations as a plain list of turns that must be searched with a vector‑retrieval (RAG) system, the author proposes that memory should be **materialized**—i.e., stored as a single, up‑to‑date, structured state that is **loaded** into the prompt on each turn.\n\nKey points of that argument:\n\n| Problem with Vector‑RAG | Why it fails | What it misses |\n|-------------------------|--------------|----------------|\n| Treats memory as a search problem | LLM has to guess which of many similar‑looking snippets is the true fact | Time, overrides, mutation, and the fact that the LLM cannot “know” which snippet is the latest |\n| No notion of state changes | The system can hallucinate outdated values (e.g., old config, old preference) | The LLM must re